# Errors, Null Safety & Resources — Polyglot Reference

How a function signals *I cannot return what you asked for* — exceptions, null sentinels, optional types — and how you guarantee cleanup of resources held during the failure.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

This notebook covers:

1. **Exceptions** — `try` / `catch` / `finally`, `throw` / `raise`, checked vs unchecked, multi-catch
2. **Null & undefined** — null type, nullable type markers, safe call, Elvis / null coalescing
3. **Optional / Result types** — wrapping the absence of a value as a value
4. **Resource management** — try-with-resources, `use`, `with`, deterministic cleanup

Concurrency-specific failure handling — `Future` failure, `Promise` rejection, async cancellation — lives in per-language repos. Pattern matching on exception types is in `08-classes-inheritance-matching.ipynb`. Statement-vs-expression position of `try` is in `03-operators-expressions.ipynb`.

## Exception Handling

The basic `try` / `catch` / `finally` shape is uniform. The divergences are around *what counts as an exception*, *whether the compiler tracks them*, and *whether `try` itself is an expression*.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| try / catch | `try { ... } catch (E e) { ... }` | `try { ... } catch { case e: E => ... }` | `try { ... } catch (e: E) { ... }` | `try { ... } catch (e) { ... }` | `try { ... } catch (e: unknown) { ... }` | `try: ... except E as e: ...` |
| throw / raise | `throw new RuntimeException(...);` | `throw new RuntimeException(...)` | `throw RuntimeException(...)` | `throw new Error(...);` | same | `raise ValueError(...)` |
| finally | `finally { ... }` | `finally { ... }` | `finally { ... }` | `finally { ... }` | same | `finally: ...` |
| multi-catch | `catch (A \| B e)` *(7+)* | pattern alternatives | multi-line catches | multi-line catches | same | `except (A, B) as e:` |
| catch any | `catch (Exception e)` | `case _: Throwable =>` | `catch (e: Throwable)` | `catch (e)` | `catch (e: unknown)` | `except:` *(or `except Exception:`)* |
| re-throw preserving stack | `throw e;` | `throw e` | `throw e` | `throw e;` | same | `raise` *(no arg)* |
| wrap with cause | `new E("ctx", e)` | `new E("ctx", e)` | `E("ctx", e)` | `new Error("ctx", { cause: e })` *(2022)* | same | `raise E(...) from e` |
| checked exceptions | yes — declare or catch | — *(JVM bytecode checked, language unchecked)* | — | — | — | — |
| `try` is an expression | — | yes — `val x = try ... catch ...` | yes | — | — | — *(statement only)* |
| exception base type | `Throwable` *(`Exception` / `Error`)* | `Throwable` | `Throwable` | (any value can be thrown) | same | `BaseException` |

Java's **checked exceptions** are unique on this list. The compiler tracks which exceptions a method can throw via the `throws` clause and forces every caller to either catch them or declare them too. Originally praised, now widely regretted — the friction made checked exceptions awkward inside lambdas and streams (you cannot throw a checked exception from a `Function.apply`), and most modern Java codebases wrap everything in `RuntimeException` to opt out.

JavaScript can throw **any value** — strings, numbers, plain objects. The convention is to throw `Error` instances for stack traces and `instanceof` checks, but the language does not enforce it. TypeScript's `catch (e: unknown)` (since 4.4) makes this explicit at the type level — older code defaulted `e` to `any`.

## Null & Undefined

The null question splits the six into three camps.

**Camp A — null is everywhere, no compiler help.** Java, JavaScript, Python (and Scala if you ignore `Option`). Any reference can be null. The compiler does not track it. NullPointerException at runtime is the norm.

**Camp B — null is opt-in via the type.** Kotlin and TypeScript-with-strict. `String` is non-nullable; `String?` (Kotlin) or `string \| null` (TypeScript) is the explicit nullable form. The compiler refuses to dereference a nullable without a check.

**Camp C — absence is a value, no special syntax.** Scala's `Option[T]` — the idiom is to use `Option` everywhere instead of nullable references. Same for Java's `Optional<T>` since 8 (though Java still has `null` for everything else).

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| absent literal | `null` | `null` *(or `None`)* | `null` | `null` *(and `undefined`!)* | same | `None` |
| nullable type marker | — *(everything nullable)* | — *(use `Option`)* | `T?` | — | `T \| null` *(strict)* | `Optional[T]` *(hint)* |
| compiler-enforced null check | — | — | yes | — | yes *(`strictNullChecks`)* | — |
| safe call | — | `opt.map(...)` | `obj?.method()` | `obj?.method()` *(2020)* | same | — |
| Elvis / null-coalescing | — | `opt.getOrElse(d)` | `a ?: b` | `a ?? b` | same | `a if a is not None else b` |
| force unwrap | — | `opt.get` *(throws)* | `obj!!` *(throws NPE)* | — | `obj!` *(non-null assertion, no runtime check)* | — |
| null check | `if (x != null)` | `if (x != null)` | `if (x != null)` *(smart-casts)* | `if (x != null)` *(also catches `undefined`)* | `if (x != null)` *(narrows type)* | `if x is not None:` |
| explicit *unset* | n/a | n/a | n/a | `undefined` | same | n/a |

**Kotlin's null safety is the cleanest design on this table.** `String` cannot be null. `String?` can. The compiler refuses `s.length` if `s: String?` — you must check (`if (s != null) s.length` smart-casts), or use safe call (`s?.length` returns `Int?`), or force unwrap (`s!!.length` throws if null). Coming from Kotlin, every other language feels under-checked.

**JavaScript's `null` vs `undefined`.** `undefined` is the absence of an assignment — uninitialized variables, missing object properties, missing function arguments, no-return functions. `null` is an explicit *this exists and is intentionally empty*. Most APIs use one or the other, rarely both. The loose-equality check `x == null` matches *both* `null` and `undefined` — one of the rare cases where double-equals is the right tool. Strict `===` distinguishes them.

## Optional / Result Types

Wrapping the absence of a value as a value. Common variants: `Option` / `Maybe` / `Optional` for presence; `Either` / `Result` / `Try` for success-or-failure with details.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| optional type | `Optional<T>` *(8+)* | `Option[T]` | — *(use `T?`)* | — | — *(use `T \| undefined`)* | `Optional[T]` *(hint = `T \| None`)* |
| present case | `Optional.of(x)` | `Some(x)` | n/a | n/a | n/a | n/a |
| absent case | `Optional.empty()` | `None` | `null` | `null` / `undefined` | same | `None` |
| map | `opt.map(f)` | `opt.map(f)` | `obj?.let(f)` | `obj?.method()` | same | manual |
| flatMap | `opt.flatMap(f)` | `opt.flatMap(f)` | `obj?.let(f)` *(if `f` returns `T?`)* | manual | same | manual |
| get-or-default | `opt.orElse(d)` | `opt.getOrElse(d)` | `x ?: d` | `x ?? d` | same | `x if x is not None else d` |
| unwrap *(unsafe)* | `opt.get()` | `opt.get` | `x!!` | (direct access) | same | (direct access) |
| filter | `opt.filter(p)` | `opt.filter(p)` | `obj?.takeIf(p)` | manual | same | manual |
| success-or-failure type | — | `Try[T]` / `Either[E, T]` | `Result<T>` | — | — | — *(use exceptions)* |

Scala is the only language here where the `Option` style is *the* idiomatic choice. Java added `Optional` in 8 with the explicit guidance that it is for *return types only* — not fields, not parameters, not collections. The Java API designers regret-publicly that `Optional` is `Serializable`-ambiguous and was sometimes used as a field type in early adopter codebases.

Kotlin deliberately did *not* add an `Option` type — its position is that nullable types `T?` plus safe calls are the same expressivity with less ceremony. Pragmatically true; the trade-off is that `T?` only models *one level* of absence — `Map<K, V?>` cannot distinguish *key not present* from *key present with null value*, which `Map<K, Option<V>>` would.

## Resource Management

Deterministic cleanup of files, sockets, locks, connections — anything where *garbage collected eventually* is not good enough. Every language here has a syntactic form for *open this, run this block, close it even if the block throws*.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| scoped-resource form | `try (Reader r = ...) { ... }` | `Using(...) { r => ... }` *(2.13+)* | `r.use { ... }` | manual `try` / `finally` | `using r = ...` *(stage 3)* | `with open(...) as f: ...` |
| cleanup interface | `AutoCloseable` | `AutoCloseable` | `Closeable` / `AutoCloseable` | manual | `Symbol.dispose` *(proposed)* | `__enter__` / `__exit__` |
| multiple resources | `try (A a = ...; B b = ...) { ... }` | nested or `Using.Manager` | nested `use { ... use { ... } }` | manual | per-spec | `with A() as a, B() as b:` |
| close ordering | reverse declaration | reverse | inner-first | manual | per-spec | reverse |
| close on exception | guaranteed | guaranteed | guaranteed | requires `finally` | guaranteed *(when impl lands)* | guaranteed |
| async cleanup | manual | manual | `use` *(suspending)* | `await using r = ...` *(proposed)* | same | `async with` |
| suppressed exceptions | `e.getSuppressed()` | similar | similar | `SuppressedError` *(proposed)* | same | `__context__` chain |

**Java's try-with-resources** (since 7) is the most widely cited example of *language feature for resource safety*. Resources go in parens, separated by semicolons; close happens in reverse declaration order; if both the body and `close()` throw, the body's exception wins and `close()`'s exception is attached as `getSuppressed()`.

**Python's `with` statement** is equivalent in expressiveness — any object with `__enter__` and `__exit__` methods is a context manager. The `contextlib` standard library provides `@contextmanager` to write context managers as generators, much terser than the explicit dunder-method form.

**Kotlin's `r.use { ... }`** is a stdlib function on `Closeable`, not a language construct. It takes a lambda and guarantees `close()` after — works on any `Closeable`. Same shape as `with` / `using`, built from a higher-order function rather than dedicated syntax.

**JavaScript** historically had only `try` / `finally`. The Stage 3 *Explicit Resource Management* proposal adds `using r = ...` for synchronous and `await using r = ...` for async — both relying on new well-known symbols `Symbol.dispose` and `Symbol.asyncDispose`. TypeScript already supports the syntax ahead of full engine adoption.

## Notes — when a cell isn't enough

**Java checked exceptions — the single most-debated language design choice.** When Java was designed, the team chose to track recoverable exceptions in the type system: methods declare `throws IOException`, callers must catch or re-declare. Intent: no `IOException` ever silently lost. In practice, three problems emerged. First, **lambdas** — `Function.apply` is not declared to throw checked exceptions, so you cannot use a method that throws `IOException` directly inside `Stream.map` without wrapping. Second, **propagation** — if a checked exception bubbles through a layer that does not know about it, you must wrap it (usually in `RuntimeException`), defeating the original intent. Third, **ergonomics** — most failures are unrecoverable; compiler-enforced ceremony to handle each one rarely improves the code. Modern Java tends toward unchecked-only — Spring deliberately wraps every checked exception. Kotlin (which targets the JVM and interops with Java) deliberately omits checked exceptions: from Kotlin's view, calling Java code that declares `throws IOException` does *not* require a `try`-catch.

**Scala `Try`, `Either`, and the railway model.** Scala provides `Try[T]` (`Success(x)` or `Failure(e: Throwable)`) and `Either[L, R]` (`Left(error)` or `Right(value)`) for *this might fail* return types. The idiomatic functional Scala style avoids exceptions entirely — operations return `Either`, and `for`-comprehensions chain them together (the *railway-oriented programming* pattern). Imperative Scala uses `try` / `catch` like Java.

**Kotlin smart casts after a null check.** Inside an `if (x != null) { ... }` block, the type of `x` narrows from `String?` to `String` automatically — you can call methods directly without `?.` or `!!`. Smart casts also work after `is` checks (`if (x is String) x.length`). The trick: smart casts only work on `val`s and stable expressions, not on mutable `var`s that another thread could change between the check and the use. For `var`s, copy to a local `val` first.

**TypeScript `strictNullChecks` — opt-in but essential.** Without `strictNullChecks`, every `T` includes `null` and `undefined` as inhabitants — the original 2012 design. With it, `T` excludes them and you must write `T | null` or `T | undefined` explicitly. Almost every modern TypeScript codebase enables it; many enable the broader `strict` flag which includes `strictNullChecks` plus a handful of others. New codebases should turn it on from day one — retrofitting onto a large codebase is painful.

**JavaScript `null` vs `undefined` — when to use which.** Conventions vary. Strongest convention: `undefined` for *no value was set*; `null` for *value was deliberately cleared*. DOM APIs return `null` for missing elements; `JSON.stringify` drops `undefined` properties but preserves `null`. The loose-equality `x == null` (intentional double-equals here) matches both — the only situation where `==` is the right choice in idiomatic JavaScript.

**Python `None` is the sole bottom value.** Unlike JavaScript's split, Python has only `None`. Comparison: prefer `x is None` over `x == None` — `is` is identity comparison, faster, and avoids surprises if the value's class overloads `__eq__`. PEP 8 mandates `is None`. Python type hints use `Optional[T]` (which is exactly `Union[T, None]`) to mark nullable parameters and return types — but type hints are not enforced at runtime; tools like `mypy` check them statically.

**Kotlin `T?` only models one level.** `String?` is *string or null*. `Map<K, V?>` is *map from K to nullable V* — but `Map<K, V?>` cannot distinguish *key not present* (returns `null` from `m[k]`) from *key present with null value* (also returns `null`). Scala's `Map[K, Option[V]]` does distinguish — the outer get returns `Option[Option[V]]`. This is the case where Kotlin's null-shortcut over `Option` loses expressivity.

**Java `Optional` — return-type only.** The Java API designers stated explicitly that `Optional<T>` is for *return values* — to make *this method may return nothing* visible in the signature. It is *not* meant for fields (it is `Serializable` but with implementation-detail semantics), parameters (just use overloads), or collection values (`Optional<T>` inside `List<Optional<T>>` is a smell). Kotlin's `T?` and Scala's `Option[T]` are more uniform — they work everywhere a regular type does. Java's restriction is a pragmatic concession to its history.

**Re-throwing without losing the stack trace.** Each language has the right way to re-throw inside a catch block such that the original stack survives. Java, Scala, Kotlin, JavaScript: `throw e;` with the same instance. Python: `raise` with no arguments — preserves the original traceback. Python's `raise NewException(...) from e` chains a new exception with the original as `__cause__`. The footgun is `throw new RuntimeException(e.getMessage())` in Java — losing both the original type and the stack. Use `throw new RuntimeException("context", e)` to wrap with cause.

**Try-with-resources close ordering.** Java closes in reverse declaration order — last opened, first closed — matching how a stack would unwind. Python's `with A() as a, B() as b:` is equivalent to nested `with` and closes B before A. Kotlin's nested `use { use { } }` is explicit. Reverse-order matters when resources have dependencies — close the connection before closing the connection pool, close the writer before closing the underlying stream.

**Suppressed exceptions.** When the body of try-with-resources throws *and* `close()` also throws, you would normally lose one. Java's `try`-with-resources keeps the body's exception as primary and attaches `close()`'s as suppressed (`e.getSuppressed()`). Python attaches the original exception as `__context__` on the new one. JavaScript's `using` proposal defines a `SuppressedError` aggregate. Without these mechanisms, the close-time error silently swallows the real cause — a famous Java 6 footgun fixed by Java 7's try-with-resources.

**`with` for non-resource use cases.** Python's `with` is sometimes used for non-resource patterns — `with mock.patch(...)`, `with open_transaction()`, `with redirect_stdout(...)`. Anything that needs setup-and-teardown around a block. The `contextlib.contextmanager` decorator turns a generator into a context manager — code before the yield is `__enter__`, code after (in `finally`) is `__exit__`. Idiomatic and underused.

**Catching too much.** All six let you catch the top of the exception hierarchy, and all six recommend against it. Python's bare `except:` catches `BaseException` — which includes `KeyboardInterrupt` and `SystemExit`, so a careless bare-except blocks Ctrl-C. Use `except Exception:` instead. Java's `catch (Throwable)` catches `OutOfMemoryError` and `StackOverflowError` — almost never what you want. Kotlin and Scala the same. JavaScript's `catch (e)` catches everything and there is no narrower form — narrow with `if (e instanceof TypeError)` inside the catch block.